In [ ]:
import os
os.chdir('..')

In [3]:
import pandas as pd
import numpy as np
from lifelines import KaplanMeierFitter, CoxPHFitter
from lifelines.statistics import logrank_test

# Reload S1
s1 = pd.read_excel('./results/proteomics/CPTAC_S1_clinical.xlsb', 
                   sheet_name=None, engine='pyxlsb')
s1a = s1['Table S1A']
s1e = s1['Table S1E']

# Tumor columns only
tumor_cols = [col for col in s1e.columns if col.startswith('T')]
hub_genes = []
for cluster in ['Cluster1_CellCycle', 'Cluster2_Immune', 'Cluster3_Epigenetic', 'Cluster4_ECM']:
    genes = pd.read_csv(f'./results/DEG/hub_gene_lists/{cluster}_hub_genes.csv')['Gene'].tolist()
    hub_genes.extend(genes)

s1e_validated = s1e[s1e['Protein/Sample.ID'].isin(hub_genes)].set_index('Protein/Sample.ID')
s1e_tumor = s1e_validated[tumor_cols]

# Survival data
survival_df = s1a[['Sample.ID', 'Status.', 'Survial.(months)']].dropna()
survival_df = survival_df.copy()
survival_df['event'] = (survival_df['Status.'] == 'dead').astype(int)
survival_df = survival_df.rename(columns={'Survial.(months)': 'time'})

# Merge
expr_T = s1e_tumor.T
expr_T.index = expr_T.index.str.replace('T', '', n=1)
expr_T.index = expr_T.index.astype(str)
survival_df['Sample.ID'] = survival_df['Sample.ID'].astype(str)
merged_df = survival_df.merge(expr_T, left_on='Sample.ID', right_index=True, how='inner')
merged_df['time'] = pd.to_numeric(merged_df['time'], errors='coerce')
merged_df['event'] = pd.to_numeric(merged_df['event'], errors='coerce')
for gene in s1e_tumor.index:
    merged_df[gene] = pd.to_numeric(merged_df[gene], errors='coerce')
merged_df = merged_df.dropna(subset=['time'])

print(f"Patients loaded: {len(merged_df)}")

Patients loaded: 104


In [7]:
from statsmodels.stats.multitest import multipletests

hr_results = pd.read_csv('./results/proteomics/km_survival_hr_results.csv')
sig_borderline_genes = hr_results[hr_results['P_value'] < 0.10]['Gene'].tolist()

# Collect raw p-values from log-rank test
pvals = []
for gene in sig_borderline_genes:
    median_val = merged_df[gene].median()
    high = merged_df[merged_df[gene] >= median_val]
    low  = merged_df[merged_df[gene] <  median_val]
    result = logrank_test(high['time'], low['time'],
                          event_observed_A=high['event'],
                          event_observed_B=low['event'])
    pvals.append(result.p_value)

# Apply Benjamini-Hochberg correction
reject, pvals_corrected, _, _ = multipletests(pvals, alpha=0.05, method='fdr_bh')

bh_df = pd.DataFrame({
    'Gene': sig_borderline_genes,
    'P_raw': np.round(pvals, 4),
    'P_corrected': np.round(pvals_corrected, 4),
    'Significant_after_correction': reject
})

print(bh_df.to_string())

      Gene   P_raw  P_corrected  Significant_after_correction
0    KIF11  0.0347       0.0818                         False
1    CENPF  0.0426       0.0818                         False
2     CDK1  0.0770       0.0847                         False
3   DLGAP5  0.0911       0.0911                         False
4    CASP1  0.0006       0.0044                          True
5     IL18  0.0008       0.0044                          True
6     TLR2  0.0116       0.0427                          True
7     TLR3  0.0669       0.0818                         False
8    CEBPB  0.0551       0.0818                         False
9    ITGB5  0.0530       0.0818                         False
10    CD44  0.0628       0.0818                         False


In [9]:
from lifelines.statistics import proportional_hazard_test

schoenfeld_results = []

for gene in sig_borderline_genes:
    median_val = merged_df[gene].median()
    temp_df = merged_df[['time', 'event', gene]].copy()
    temp_df['group'] = (temp_df[gene] >= median_val).astype(int)
    
    cph = CoxPHFitter()
    cph.fit(temp_df[['time', 'event', 'group']], 
            duration_col='time', event_col='event')
    
    result = proportional_hazard_test(cph, temp_df[['time', 'event', 'group']], 
                                      time_transform='rank')
    
    schoenfeld_results.append({
        'Gene': gene,
        'Test_statistic': round(result.summary['test_statistic']['group'], 4),
        'P_value': round(result.summary['p']['group'], 4),
        'PH_assumption_met': 'Yes' if result.summary['p']['group'] > 0.05 else 'Violated'
    })

schoenfeld_df = pd.DataFrame(schoenfeld_results)
print(schoenfeld_df.to_string())

      Gene  Test_statistic  P_value PH_assumption_met
0    KIF11          0.7599   0.3834               Yes
1    CENPF          0.0180   0.8933               Yes
2     CDK1          1.0796   0.2988               Yes
3   DLGAP5          0.0003   0.9857               Yes
4    CASP1          1.5245   0.2169               Yes
5     IL18          1.5968   0.2064               Yes
6     TLR2          0.7792   0.3774               Yes
7     TLR3          2.3583   0.1246               Yes
8    CEBPB          2.5837   0.1080               Yes
9    ITGB5          2.6157   0.1058               Yes
10    CD44          2.8806   0.0897               Yes


In [11]:
# Save Benjamini-Hochberg results
bh_df.to_csv('./results/proteomics/bh_correction_results.csv', index=False)

# Save Schoenfeld results
schoenfeld_df.to_csv('./results/proteomics/schoenfeld_ph_test_results.csv', index=False)

print("Both saved.")

Both saved.
